In [0]:
%run ../init_framework

In [0]:
# Initialize the CDF Read Stream ---
df_bronze_repayments = (spark.readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) 
    .table(REPAYMENTS_BRONZE))

In [0]:
# --- 1. Define Rename Map ---
repayments_rename_map = {
    "total_rec_prncp": "total_principal_received",
    "total_rec_int": "total_interest_received",
    "total_rec_late_fee": "total_late_fee_received",
    "total_pymnt": "total_payment_received",
    "last_pymnt_amnt": "last_payment_amount",
    "last_pymnt_d": "last_payment_date",
    "next_pymnt_d": "next_payment_date",
}

# --- 2. Execute Transformation ---
df_renamed_repayments = standardize_column_names(
    df_bronze_repayments, repayments_rename_map, strict=True
)

In [0]:
df_with_metadata = apply_silver_metadata(df_renamed_repayments)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
# Keep Valid Repayments: Drop NULL or UNKNOWN
col_ls = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received", "last_payment_amount"]
df_valid_repayments = drop_critical_nulls(df_distinct, col_ls)

In [0]:
from pyspark.sql.functions import col, when

# Apply repair logic to total_payment_received
df_payment_fixed = df_valid_repayments.withColumn(
    "total_payment_received",
    when(
        (col("total_principal_received") != 0.0)
        & (col("total_payment_received") == 0.0),
        col("total_principal_received")
        + col("total_interest_received")
        + col("total_late_fee_received"),
    ).otherwise(col("total_payment_received")),
)

In [0]:
df_valid_payment = df_payment_fixed.filter("total_payment_received != 0.0")


In [0]:
from pyspark.sql.functions import col, when, lit

# Replace string "0.0" with a true NULL/None
df_last_payments_fixed = df_valid_payment.withColumn(
  "last_payment_date",
   when(
       col("last_payment_date") == "0.0",
       lit(None).cast("string") 
   ).otherwise(col("last_payment_date"))
)

In [0]:
# df_last_payments_fixed.filter("last_payment_date is NULL").count()

In [0]:
from pyspark.sql.functions import col, when, lit

# Replace string "0.0" with a true NULL/None
df_last_payments_fixed = df_valid_payment.withColumn(
  "next_payment_date",
   when(
       col("next_payment_date") == "0.0",
       lit(None).cast("string") 
   ).otherwise(col("next_payment_date"))
)

In [0]:
# Final deduplication on the primary key to prevent MERGE conflicts
df_repayments_final = df_last_payments_fixed.dropDuplicates(["loan_id"])

In [0]:
def upsert_repayments_to_silver(microBatchDF, batchId):
    """
    Stateless MERGE from Bronze CDF into Silver Repayments using explicit SQL mapping.
    """
    spark_session = microBatchDF.sparkSession

    # Register the incoming micro-batch as a Temp View
    microBatchDF.createOrReplaceTempView("repayments_batch_updates")

    # Execute explicit SQL MERGE. 
    # Joined strictly on loan_id. 
    merge_query = f"""
    MERGE INTO {REPAYMENTS_SILVER} AS target
        USING repayments_batch_updates AS source
        ON target.loan_id = source.loan_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.total_principal_received = source.total_principal_received,
            target.total_interest_received = source.total_interest_received,
            target.total_late_fee_received = source.total_late_fee_received,
            target.total_payment_received = source.total_payment_received,
            target.last_payment_amount = source.last_payment_amount,
            target.last_payment_date = source.last_payment_date,
            target.next_payment_date = source.next_payment_date,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at
        WHEN NOT MATCHED THEN
          INSERT (
            loan_id, total_principal_received, total_interest_received, total_late_fee_received,
            total_payment_received, last_payment_amount, last_payment_date, next_payment_date,
            _ingested_at, _source_file, _bronze_version, _updated_at
          )
          VALUES (
            source.loan_id, source.total_principal_received, source.total_interest_received, source.total_late_fee_received,
            source.total_payment_received, source.last_payment_amount, source.last_payment_date, source.next_payment_date,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at
          )
    """
    spark_session.sql(merge_query)
 
# Configure shuffle partitions for local cluster core count
spark.conf.set("spark.sql.shuffle.partitions", "16")
 
# --- Start Streaming Pipeline ---
query_repayments = (
    df_repayments_final.writeStream
    .outputMode("append") # Required for stateless CDF reads
    .option("checkpointLocation", SILVER_CHECKPOINT_REPAYMENTS)
    .trigger(availableNow=True)
    .foreachBatch(upsert_repayments_to_silver)
    .start()
)

# Block cluster exit until this stream finishes
query_repayments.awaitTermination()

In [0]:
%sql
select * from lending_risk.silver.repayments
limit 3;
-- select count(1) from lending_risk.silver.repayments;